# Fine-tuning an Open-source Transformer for Financial Sentiment Classification

We fine-tune an open-source transformer model on labelled financial sentences from the Hugging Face dataset `atrost/financial_phrasebank`. The task is three-class sentiment classification:

- `negative`
- `neutral`
- `positive`

The default model is `distilbert/distilbert-base-uncased`, which is small enough for a live classroom demo.

## Learning Objectives

By the end of this notebook, students should understand:

1. What fine-tuning means.
2. How labelled text data are used to adapt a general language model to a finance-specific task.
3. The difference between a pretrained model and a fine-tuned classifier.
4. How to evaluate a fine-tuned model using accuracy, F1 score, and a confusion matrix.
5. Why fine-tuning is useful for repeated classification tasks but less suitable for complex reasoning/extraction tasks.

## Colab Data Requirements

This notebook does **not** require any raw CSV files, local folders, Google Drive mounts, or API keys. When opened from GitHub in Colab, it downloads the labelled dataset from Hugging Face (`atrost/financial_phrasebank`) and downloads the selected model weights from Hugging Face.


## 1. Setup

This notebook is designed for Google Colab. It installs the required packages, imports the main libraries, and fixes random seeds for reproducibility.

Packages used:

- `transformers`: Provides state-of-the-art pretrained models for various NLP tasks.
- `datasets`: Offers a lightweight library to easily access and share datasets for NLP, computer vision, and audio tasks.
- `evaluate`: A Hugging Face library for easily comparing different models and experiments with various metrics.
- `accelerate`: Simplifies the process of running PyTorch models on different hardware configurations (e.g., multiple GPUs, TPUs).
- `scikit-learn`: A popular machine learning library offering tools for data mining and data analysis.
- `torch`: The PyTorch deep learning framework, used for building and training neural networks.

In [1]:
!pip -q install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [2]:
import os
import random
import inspect

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["WANDB_DISABLED"] = "true"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Explain the Model

We use `distilbert/distilbert-base-uncased` as the default open-source model.

DistilBERT is a smaller, faster version of BERT. It is suitable for a live classroom demo because it trains more quickly than full BERT while still illustrating the transformer fine-tuning workflow.

Optional finance-specific backbone:

```python
MODEL_NAME = "yiyanghkust/finbert-pretrain"
```

`yiyanghkust/finbert-pretrain` is pretrained on financial communication text, including 10-K/10-Q filings, earnings calls, and analyst reports. It may be more finance-aware, but we keep DistilBERT as the default for speed and reliability in class.

## 3. Load Data

We use the Hugging Face dataset `atrost/financial_phrasebank`.

The dataset has financial sentences labelled as negative, neutral, or positive. Different dataset configurations/splits may expose labels as strings or integers, so the code below is written defensively.

In [3]:
DATASET_NAME = "atrost/financial_phrasebank"

raw = load_dataset(DATASET_NAME)
print(raw)
print("Available splits:", list(raw.keys()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/721 [00:00<?, ?B/s]

data/train-00000-of-00001-138b53eb17a3e8(…):   0%|          | 0.00/268k [00:00<?, ?B/s]

data/validation-00000-of-00001-0876be41e(…):   0%|          | 0.00/68.7k [00:00<?, ?B/s]

data/test-00000-of-00001-41c7ea948573445(…):   0%|          | 0.00/82.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/776 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})
Available splits: ['train', 'validation', 'test']


In [4]:
# Some Hugging Face datasets provide train/validation/test splits.
# If a dataset only provides one split, create train/validation/test splits.
if {"train", "validation", "test"}.issubset(set(raw.keys())):
    dataset = DatasetDict({
        "train": raw["train"],
        "validation": raw["validation"],
        "test": raw["test"],
    })
else:
    first_split = raw[list(raw.keys())[0]]
    split_1 = first_split.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label" if "label" in first_split.column_names else None)
    split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED, stratify_by_column="label" if "label" in split_1["test"].column_names else None)
    dataset = DatasetDict({
        "train": split_1["train"],
        "validation": split_2["train"],
        "test": split_2["test"],
    })

# User requested validation sentences to 300, rest for test.
# Combine existing validation and test sets for re-splitting
from datasets import concatenate_datasets
combined_val_test = concatenate_datasets([dataset["validation"], dataset["test"]])

new_validation_size = 300

# Determine column for stratification. 'label' should be present at this stage.
stratify_col = None
if "label" in combined_val_test.column_names:
    stratify_col = "label"

# Perform the split: new_splits['train'] will be the new validation, new_splits['test'] will be the new test.
new_splits = combined_val_test.train_test_split(
    train_size=new_validation_size, # This is the absolute number of examples for the 'train' part of this split (new validation)
    shuffle=True,
    seed=SEED,
    stratify_by_column=stratify_col
)

dataset["validation"] = new_splits["train"]
dataset["test"] = new_splits["test"]

print("Updated dataset splits after re-splitting validation and test:")
print(dataset)


label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

# Detect text and label columns.
possible_text_cols = ["sentence", "text", "content"]
TEXT_COL = next((c for c in possible_text_cols if c in dataset["train"].column_names), None)
if TEXT_COL is None:
    raise ValueError(f"Could not find text column. Columns are: {dataset['train'].column_names}")

possible_label_cols = ["label", "labels", "sentiment"]
LABEL_COL = next((c for c in possible_label_cols if c in dataset["train"].column_names), None)
if LABEL_COL is None:
    raise ValueError(f"Could not find label column. Columns are: {dataset['train'].column_names}")

print("Text column:", TEXT_COL)
print("Original label column:", LABEL_COL)

# Robust label mapping: labels may be integers, ClassLabel objects, or strings.
label_feature = dataset["train"].features.get(LABEL_COL)
class_names = getattr(label_feature, "names", None)
print("Dataset class names:", class_names)

# Handle possible class-name ordering differences defensively.
def normalize_label(example):
    value = example[LABEL_COL]
    if isinstance(value, str):
        label_name = value.strip().lower()
    elif class_names is not None:
        label_name = class_names[int(value)].strip().lower()
    else:
        # Most Financial PhraseBank versions use 0=negative, 1=neutral, 2=positive.
        label_name = id2label[int(value)]
    example["labels"] = label2id[label_name]
    return example

dataset = dataset.map(normalize_label)

# Keep the original text column name, but create a consistent "sentence" column if needed.
if TEXT_COL != "sentence":
    dataset = dataset.rename_column(TEXT_COL, "sentence")
    TEXT_COL = "sentence"

print(dataset)
print("Final columns:", dataset["train"].column_names)

Updated dataset splits after re-splitting validation and test:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 300
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 1446
    })
})
Text column: sentence
Original label column: label
Dataset class names: ['negative', 'neutral', 'positive']


Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1446 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 300
    })
    test: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 1446
    })
})
Final columns: ['sentence', 'label', 'labels']


In [5]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

# Detect text and label columns.
possible_text_cols = ["sentence", "text", "content"]
TEXT_COL = next((c for c in possible_text_cols if c in dataset["train"].column_names), None)
if TEXT_COL is None:
    raise ValueError(f"Could not find text column. Columns are: {dataset['train'].column_names}")

possible_label_cols = ["label", "labels", "sentiment"]
LABEL_COL = next((c for c in possible_label_cols if c in dataset["train"].column_names), None)
if LABEL_COL is None:
    raise ValueError(f"Could not find label column. Columns are: {dataset['train'].column_names}")

print("Text column:", TEXT_COL)
print("Original label column:", LABEL_COL)

# Robust label mapping: labels may be integers, ClassLabel objects, or strings.
label_feature = dataset["train"].features.get(LABEL_COL)
class_names = getattr(label_feature, "names", None)
print("Dataset class names:", class_names)

# Handle possible class-name ordering differences defensively.
def normalize_label(example):
    value = example[LABEL_COL]
    if isinstance(value, str):
        label_name = value.strip().lower()
    elif class_names is not None:
        label_name = class_names[int(value)].strip().lower()
    else:
        # Most Financial PhraseBank versions use 0=negative, 1=neutral, 2=positive.
        label_name = id2label[int(value)]
    example["labels"] = label2id[label_name]
    return example

dataset = dataset.map(normalize_label)

# Keep the original text column name, but create a consistent "sentence" column if needed.
if TEXT_COL != "sentence":
    dataset = dataset.rename_column(TEXT_COL, "sentence")
    TEXT_COL = "sentence"

print(dataset)
print("Final columns:", dataset["train"].column_names)

Text column: sentence
Original label column: label
Dataset class names: ['negative', 'neutral', 'positive']


Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1446 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 300
    })
    test: Dataset({
        features: ['sentence', 'label', 'labels'],
        num_rows: 1446
    })
})
Final columns: ['sentence', 'label', 'labels']


In [6]:
# Quick pandas table for label counts.
train_df = dataset["train"].to_pandas()
train_df["label_name"] = train_df["labels"].map(id2label)
train_df["label_name"].value_counts().rename_axis("label").reset_index(name="count")

,label,count
0,neutral,1852
1,positive,866
2,negative,382


## 4. Quick Data Inspection

The input is one financial sentence.

The output is one of three sentiment labels: negative, neutral, or positive.

This is supervised classification, not text generation. The model learns from labelled examples.

In [7]:
for split in ["train", "validation", "test"]:
    df = dataset[split].to_pandas()
    df["label_name"] = df["labels"].map(id2label)
    print(f"\n{split.upper()} class balance")
    print(df["label_name"].value_counts().sort_index())

print("\nFive random training examples:")
sample_df = train_df.sample(5, random_state=SEED)[["sentence", "label_name"]]
display(sample_df)


TRAIN class balance
label_name
negative     382
neutral     1852
positive     866
Name: count, dtype: int64

VALIDATION class balance
label_name
negative     38
neutral     177
positive     85
Name: count, dtype: int64

TEST class balance
label_name
negative    184
neutral     850
positive    412
Name: count, dtype: int64

Five random training examples:


,sentence,label_name
718,This combined with foreign investments creates...,positive
2953,The study was not designed to enable formal st...,neutral
1805,"In Finland , OP-Pohjola 's staff union is boyc...",negative
1612,"The Costanza light , with an aluminum base and...",neutral
1190,"BG AD , Bulgaria 's leading Internet company .",positive


## 5. Load Tokenizer and Model

We load the tokenizer and a sequence-classification model with three output labels.

The base language model is pretrained. The classification head is task-specific and will be adapted during fine-tuning.

**General guidelines:**



*   num_train_epochs=2: This is set to a very small number (2) primarily for demonstration purposes, especially in a live Colab environment where training time needs to be kept short. For real-world applications, fine-tuning often requires more epochs, but it's important to monitor validation performance to avoid overfitting. Typically, you might see anywhere from 3 to 10 epochs, but this depends heavily on the dataset size and task complexity.
*   learning_rate=2e-5: This is a common starting point for fine-tuning pre-trained transformer models. Since the model is already pre-trained and has learned good general language representations, you typically want a small learning rate to fine-tune it subtly rather than drastically changing its weights. A reasonable range is often 1e-5 to 5e-5. Higher learning rates risk destabilizing the pre-trained weights, while much lower rates might make training too slow.
*   per_device_train_batch_size=16 and per_device_eval_batch_size=32: These control how many examples are processed at once. The training batch size is often chosen to be smaller than the evaluation batch size to manage GPU memory, especially during backpropagation. Smaller batch sizes (e.g., 8, 16, 32) can sometimes lead to better generalization, but larger batch sizes (e.g., 64, 128) can speed up training if memory allows. The choice depends on available GPU memory and how noisy the gradient estimates can be.
*   weight_decay=0.01: This is a regularization technique that helps prevent overfitting by penalizing large weights. It's a standard practice in deep learning. A typical range for weight_decay is between 0.001 and 0.1.
*   save_strategy="epoch", logging_steps=20, load_best_model_at_end=True, metric_for_best_model="f1_macro": These are more about managing the training process and evaluating the model. Saving per epoch, logging frequently, and loading the best model based on a chosen metric (like F1-macro, which is good for imbalanced datasets) are all good practices.


**All these parameters can depend significantly on:**

*   Task Complexity: More complex tasks (e.g., nuanced reasoning vs. simple classification) might benefit from more epochs or different learning rates.
*   Training Data Size: If you have a very large dataset, you might need fewer epochs or can tolerate larger batch sizes. For small datasets, more aggressive regularization (like higher weight_decay) or earlier stopping might be necessary to prevent overfitting.
*   Sentence Length (MAX_LENGTH): While MAX_LENGTH is specifically for tokenization and model input, it indirectly affects batch size. Longer sequences consume more GPU memory, which might necessitate smaller per_device_train_batch_size and per_device_eval_batch_size to avoid out-of-memory errors.
*   Model Size: Larger models generally require more careful tuning of learning rates and batch sizes due to their complexity and memory footprint.






In [8]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
# Optional finance-specific backbone, slower but finance-aware:
# MODEL_NAME = "yiyanghkust/finbert-pretrain"

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

print("Loaded model:", MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model: distilbert/distilbert-base-uncased


This report provides information about how the weights from the pre-trained distilbert/distilbert-base-uncased model were loaded into your DistilBertForSequenceClassification model. Here's what each status means:

UNEXPECTED: This refers to layers like vocab_transform, vocab_layer_norm, and vocab_projector. These are typically part of the original pre-training head (e.g., for Masked Language Modeling or Next Sentence Prediction) of DistilBERT. When you load DistilBertForSequenceClassification, you're switching to a new task (sequence classification), so these pre-training specific layers are no longer needed or expected. Their weights are dropped, which is normal and can be ignored because they are not used in your new model architecture.

MISSING: This refers to the classifier and pre_classifier layers. These are the weights for the new classification head that is added on top of the pre-trained DistilBERT body for your sentiment classification task. Since these layers are specific to your downstream task and were not part of the original pre-trained DistilBERT, their weights are MISSING from the loaded checkpoint. This is also expected behavior. These MISSING parameters will be randomly initialized, and then updated during the fine-tuning process as the model learns to classify financial sentiment.

## 6. Tokenize Data

Transformers do not consume raw strings directly. We tokenize each sentence into token IDs and attention masks.

Padding is handled dynamically by `DataCollatorWithPadding`.

In [9]:
def tokenize_function(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove columns that Trainer does not need. Keep labels and tokenized fields.
keep_cols = {"input_ids", "attention_mask", "labels"}
for split in tokenized_dataset:
    remove_cols = [c for c in tokenized_dataset[split].column_names if c not in keep_cols]
    tokenized_dataset[split] = tokenized_dataset[split].remove_columns(remove_cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized_dataset)

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1446 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 300
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1446
    })
})


## 7. Baseline Before Fine-tuning

Before fine-tuning, the transformer body is pretrained, but the classification head has not yet learned this three-class financial sentiment task.

Therefore, predictions from the newly initialized classification head are not meaningful yet.

The output you're seeing is from the model before fine-tuning. At this stage, the classification head has been randomly initialized and hasn't learned to classify financial sentiment yet. That's why the probabilities for 'negative', 'neutral', and 'positive' for each sentence are all very close to 0.333 (1/3). The model is essentially guessing randomly, which often results in a mix of predictions, including neutral, negative, and positive, but none of them are reliable at this point.

In [10]:
def predict_examples(model, tokenizer, sentences, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    encoded = tokenizer(sentences, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=1)
    return pd.DataFrame({
        "sentence": sentences,
        "prediction": [id2label[int(i)] for i in pred_ids],
        "prob_negative": probs[:, 0],
        "prob_neutral": probs[:, 1],
        "prob_positive": probs[:, 2],
    })

baseline_sentences = [dataset["test"][i]["sentence"] for i in range(5)]
predict_examples(model, tokenizer, baseline_sentences)

,sentence,prediction,prob_negative,prob_neutral,prob_positive
0,Neste Shipping is the most likely to remain Fi...,neutral,0.326822,0.339108,0.334070
1,Finnish Suominen Flexible Packaging is cutting...,positive,0.332041,0.324707,0.343252
2,Other potential clients include public adminis...,negative,0.343908,0.325022,0.331071
3,Cablevision Systems Corp. CVC Their Madison Sq...,neutral,0.320826,0.340417,0.338757
4,"Furthermore , sales of new passenger cars and ...",neutral,0.325907,0.337872,0.336220


## 8. Fine-tune Model

Fine-tuning updates the model parameters using labelled examples from the financial sentiment dataset.

During fine-tuning, all the parameters of the DistilBertForSequenceClassification model are updated. This includes:

The parameters of the newly added classification head: These are the classifier.bias, pre_classifier.bias, classifier.weight, and pre_classifier.weight that were initially MISSING and randomly initialized. These will undergo significant updates to learn the financial sentiment classification task.
The parameters of the pre-trained DistilBERT body: Although these parameters were loaded from a pre-trained model, they are also updated during fine-tuning. The updates are typically smaller compared to the classification head, as the pre-trained body has already learned general language representations. The fine-tuning process adapts these general representations to be more specific and useful for the financial sentiment task.

We train for only two epochs to keep the runtime manageable for a live Colab demo.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

base_training_args = dict(
    output_dir="./distilbert_financial_phrasebank",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    seed=SEED,
)

# Transformers changed the argument name from evaluation_strategy to eval_strategy.
# Try the newer name first; fall back for older versions.
try:
    training_args = TrainingArguments(eval_strategy="epoch", **base_training_args)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **base_training_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

CUDA available: True
GPU: Tesla T4


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.528866,0.555547,0.796667,0.756426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=194, training_loss=0.6848500635206085, metrics={'train_runtime': 25.8294, 'train_samples_per_second': 120.018, 'train_steps_per_second': 7.511, 'total_flos': 47348625286008.0, 'train_loss': 0.6848500635206085, 'epoch': 1.0})

## 9. Evaluate on Held-out Test Set

The test set was not used for training. It gives us a cleaner estimate of how the fine-tuned classifier performs on unseen labelled data.

Macro F1 is useful when class sizes differ because it gives equal weight to each class rather than letting the largest class dominate the score.

In [12]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])
print(test_metrics)

pred_output = trainer.predict(tokenized_dataset["test"])
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="macro")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("Accuracy:", round(acc, 4))
print("Macro F1:", round(f1, 4))
print("\nConfusion matrix, rows=true labels, columns=predicted labels")
print(pd.DataFrame(cm, index=["true_negative", "true_neutral", "true_positive"], columns=["pred_negative", "pred_neutral", "pred_positive"]))
print("\nClassification report")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.528866,0.546480,1,0.793223,0.761004


{'eval_loss': 0.5464798212051392, 'eval_accuracy': 0.793222683264177, 'eval_f1_macro': 0.7610036098295935}


Accuracy: 0.7932
Macro F1: 0.761

Confusion matrix, rows=true labels, columns=predicted labels
               pred_negative  pred_neutral  pred_positive
true_negative            139            29             16
true_neutral              24           742             84
true_positive             25           121            266

Classification report
              precision    recall  f1-score   support

    negative       0.74      0.76      0.75       184
     neutral       0.83      0.87      0.85       850
    positive       0.73      0.65      0.68       412

    accuracy                           0.79      1446
   macro avg       0.77      0.76      0.76      1446
weighted avg       0.79      0.79      0.79      1446



## 10. Test on New Finance Examples

Now we classify new financial sentences using the fine-tuned model.

In [14]:
new_examples = [
    "Operating profit decreased to EUR 11.2 million from EUR 16.6 million.",
    "EBIT margin was up from 1.4 percent to 5.1 percent.",
    "The company announced a new share repurchase programme.",
    "The firm warned that weak demand will reduce revenue next quarter.",
    "The acquisition is expected to close by the end of August.",
]

# Use the predict_examples function which already works as intended
# and returns a DataFrame with all necessary probability columns.
predictions_df = predict_examples(model, tokenizer, new_examples)

# The result is already in the desired DataFrame format
# so we can just display it directly.
pd.DataFrame(predictions_df)

,sentence,prediction,prob_negative,prob_neutral,prob_positive
0,Operating profit decreased to EUR 11.2 million...,negative,0.598853,0.090406,0.310741
1,EBIT margin was up from 1.4 percent to 5.1 per...,positive,0.268910,0.179461,0.551629
2,The company announced a new share repurchase p...,neutral,0.039001,0.671678,0.289321
3,The firm warned that weak demand will reduce r...,negative,0.510559,0.146397,0.343044
4,The acquisition is expected to close by the en...,neutral,0.070146,0.767748,0.162106


## 11. Save Model Locally

The fine-tuned model can be saved and reused later to classify new financial sentences.

This does not push anything to Hugging Face Hub and does not require login.

In [15]:
SAVE_DIR = "./fine_tuned_financial_sentiment_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

predict_examples(reloaded_model, reloaded_tokenizer, [new_examples[0]])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./fine_tuned_financial_sentiment_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,sentence,prediction,prob_negative,prob_neutral,prob_positive
0,Operating profit decreased to EUR 11.2 million...,negative,0.598853,0.090406,0.310741


## 12. Teaching Discussion

### What has the model learned?

It learned to map financial sentences to negative / neutral / positive labels.

### What does it not do?

It does not reason, extract causal mechanisms, or generate explanations. It is a classifier trained to predict one label per sentence.

### When is this useful?

Fine-tuning is useful for repeated classification at scale, especially when the task is well-defined and labelled examples exist.

### When is frontier LLM + RAG better?

A frontier LLM plus retrieval-augmented generation is often better when the task requires complex reasoning, structured extraction, or using context spread across multiple passages.

### Connection to Li et al. 2026

This fine-tuning demo teaches the classifier part of the NLP toolkit. Li et al. use a hybrid workflow: filtering/retrieval with traditional NLP or transformer tools, then frontier LLM extraction for culture type, tone, causes, effects, and causal triples.

## 13. Optional Extension: Use FinBERT-pretrain

Do not run this during the main demo unless you have extra time.

To switch to a finance-specific backbone, restart the runtime and change the model name near the top of the notebook:

```python
MODEL_NAME = "yiyanghkust/finbert-pretrain"
```

This model is finance-specific but may be slower than DistilBERT. It is useful for discussing how domain pretraining differs from task fine-tuning.

In [16]:
# Optional extension only. Do not run by default.
# MODEL_NAME = "yiyanghkust/finbert-pretrain"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=3,
#     id2label=id2label,
#     label2id=label2id,
# )